###

# import torch

torch is the main library of PyTorch.

It provides tools for tensor operations, GPU computing, and deep learning.

Why used?
Transformers perform many matrix operations, and PyTorch tensors handle them efficiently.

# import torch.nn as nn

torch.nn contains all neural network components like layers, loss functions, etc.

as nn is just a short alias.

Example components:

Linear layers

Embeddings

LayerNorm

Transformer layers

# import torch.nn.functional as F

Contains activation functions and operations that don't require parameters.

Examples:

softmax

relu

sigmoid

Why used?

In attention we use:

softmax

In [6]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

In [56]:
class SimpleSelfAttention(nn.Module):
    def __init__(self,embed_dim):
        super().__init__()
        self.embed_dim=embed_dim
        """
           embed_dim

           Embedding dimension.

           Example:

           embed_dim = 512

           Meaning every token is represented by a 512-dimensional vector."""
        self.W_q=nn.Linear(embed_dim,embed_dim)
        
        self.W_k=nn.Linear(embed_dim,embed_dim)
        
        self.W_v=nn.Linear(embed_dim,embed_dim)
        """
                    Why Q, K, V?
            
            Self attention works using:
            
            Attention(Q,K,V)
            
            Meaning:
            
            Query asks a question
            
            Key matches the query
            
            Value contains the information
            
            Example:
            
            Sentence:
            
            "The animal didn't cross the street because it was tired"
            
            Attention helps "it" focus on "animal".
        """
    def forward(self,x):
        Q=self.W_q(x)
        K=self.W_k(x)
        V=self.W_v(x)
        """
        Creating Query, Key, Value
        
        Q=self.W_q(x)
        
        Transforms input into Query vectors.
        
        K=self.W_k(x)
        
        Transforms input into Key vectors.
        
        V=self.W_v(x)
        
        Transforms input into Value vectors.
        """
        scores=torch.matmul(Q,K.transpose(-2,-1))/(self.embed_dim**0.5)

        """
                torch.matmul()
        
        Matrix multiplication.
        
        Used to calculate similarity between queries and keys.
        
        K.transpose(-2,-1)
        
        Swaps last two dimensions.
        
        Why?
        
        Matrix multiplication requires shape alignment.
        
        Example:
        
        Q : (seq_len , embed_dim)
        K : (embed_dim , seq_len)
        Scaling
        (self.embed_dim**0.5)
        
        Which means:
        
        sqrt(embed_dim)
        
        Why?
        
        Without scaling:
        
        dot product becomes large
        
        softmax becomes unstable
        
        So transformer paper introduced scaled dot-product attention.
                """
        weights=F.softmax(scores,dim=-1)

        """
                softmax
        
        Converts scores into probabilities.
        
        Example:
        
        [3.2,1.2,0.5]
        
        becomes
        
        [0.7,0.2,0.1]
        
        Meaning:
        
        Token attends 70% to first token.
        
        dim=-1
        
        Apply softmax across the last dimension (tokens).
        """
        output=torch.matmul(weights,V)

        """
                Computes weighted average of value vectors.
        
        Meaning:
        
        Each token gathers information from important tokens.
        """

        return output

In [57]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self,embed_dim,num_heads,ff_dim,dropout=0.1):
        """
             Parameters
            
            embed_dim
            
            Dimension of embeddings.
            
            Example:
            
            512
            
            num_heads
            
            Number of attention heads.
            
            Example:
            
            8 heads
            
            Each head learns different relationships.
            
            ff_dim
            
            Hidden size of feed-forward network.
            
            Example:
            
            2048
            
            dropout
            
            Regularization to avoid overfitting.
            
            Default:
            
            0.1
            
            Meaning 10% neurons randomly ignored.
                    """
        super().__init__()
        self.attention=nn.MultiheadAttention(emded_dim,num_heads)
        """
                Built-in PyTorch implementation.
        
        Instead of single attention, transformer uses multiple heads.
        
        Example:
        
        8 heads learn:
        
        syntax
        
        semantics
        
        dependency relations
        """
        self.norm1=nn.LayerNorm(embed_dim)

        self.ff=nn.Sequential(
            nn.Linear(embed_dim,ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim,emded_dim)
        )

        self.norm2=nn.LayerNorm(embed_dim)
        self.dropout=nn.Dropout(dropout)
        """
        Randomly drops neurons.

        Helps generalization.
        """
    def forward(self,x):
        attn_output,_=self.attention(x,x,x)
        x=self.norm1(x+self.dropout(attn_output))
        ff_output=self.ff(x)
        x=self.norm2(x+self.dropout(ff_output))

        return x
        

In [9]:
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, max_len):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        """
                Converts tokens → vectors.
        
        Example:
        
        "cat" → [0.23, -0.41, ...]
        """
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, embed_dim))

        """
                Transformers do not understand word order.
        
        So we add position information.
        
        Example:
        
        word position 1
        word position 2
         """

        self.layers = nn.ModuleList([
            TransformerEncoderLayer(embed_dim, num_heads, ff_dim)
            for _ in range(num_layers)
        ])
        """
                Creates multiple encoder blocks.
        
        Example:
        
        num_layers = 6
        """
        self.fc_out = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)

        x = self.embedding(x) + self.pos_embedding[:, :seq_len, :]

        x = x.transpose(0, 1)  # required by MultiheadAttention

        for layer in self.layers:
            x = layer(x)

        x = x.transpose(0, 1)
        logits = self.fc_out(x)

        return logits

###

# PyTorch provides full transformer architecture.

Parameters:

d_model=512

Embedding size.

nhead=8

Number of attention heads.

num_encoder_layers=6

Number of encoder layers.

num_decoder_layers=6

Number of decoder layers.

Used for tasks like:

translation

text generation

dim_feedforward=2048

Hidden size of feed-forward network.

dropout=0.1

Regularization.

In [10]:
model = nn.Transformer(
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1
)

In [11]:
# Create dummy input
src = torch.rand(10, 32, 512)  # source sequence
tgt = torch.rand(20, 32, 512)  # target sequence

# Run model
output = model(src, tgt)

# Print output
print(output.shape)

torch.Size([20, 32, 512])


In [12]:
print(output)

tensor([[[ 0.0382, -0.4436,  0.9327,  ...,  0.3633, -0.3797, -0.9324],
         [ 1.0356, -1.0648,  0.9893,  ...,  0.3065, -0.4574, -0.6258],
         [ 0.1772,  0.6332,  1.0307,  ...,  0.9075, -0.8907, -0.9049],
         ...,
         [ 0.4647, -1.1442,  1.0309,  ...,  0.9406, -1.2207, -1.3658],
         [ 0.0518,  0.0814,  1.1372,  ...,  0.3506, -0.9186, -0.2155],
         [ 0.3419, -0.7160,  1.8850,  ...,  0.9037, -0.5277, -0.8240]],

        [[-0.2088, -1.3034,  0.8551,  ...,  0.4653,  0.0897, -0.7054],
         [ 0.1720, -0.4600,  1.2136,  ...,  0.2768, -0.4346, -1.4277],
         [ 0.2838, -0.9782,  1.1324,  ...,  0.4325, -0.5821, -0.8433],
         ...,
         [ 0.6432,  0.1724,  1.2387,  ...,  0.5194, -1.1859, -0.8957],
         [ 0.0570, -0.3174,  0.5623,  ...,  0.4871, -0.9763, -0.3828],
         [ 0.2122, -0.4659,  1.1346,  ...,  0.5591,  0.0433, -1.6167]],

        [[-0.0162, -0.1272,  1.4755,  ...,  0.5033, -1.0329, -0.8680],
         [-0.1470, -0.1246,  1.2152,  ...,  0

###Text Data

   ↓
Tokenization

   ↓
   
Word

→ Index

   ↓
   
Create Sequences

   ↓
Embedding Layer

   ↓
Transformer Encoder/Decoder

   ↓
Linear Layer

   ↓
Softmax

   ↓
Loss (CrossEntropy)

   ↓
Backpropagation

   ↓
Update Weights


In [41]:
text = "hello world hello transformer hello attention"

In [42]:
words = text.split()
vocab = list(set(words))

In [43]:
word_to_idx = {word:i for i,word in enumerate(vocab)}
idx_to_word = {i:word for word,i in word_to_idx.items()}

In [44]:
tokens = [word_to_idx[word] for word in words]

In [45]:
seq_length = 3
inputs = []
targets = []

for i in range(len(tokens)-seq_length):
    inputs.append(tokens[i:i+seq_length])
    targets.append(tokens[i+1:i+seq_length+1])

In [46]:
inputs = torch.tensor(inputs)
targets = torch.tensor(targets)

In [47]:

import torch
import torch.nn as nn
import torch.optim as optim

class SimpleTransformer(nn.Module):

    def __init__(self,vocab_size,embed_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size,embed_dim)

        self.transformer = nn.Transformer(
            d_model=embed_dim,
            nhead=2,
            num_encoder_layers=2,
            num_decoder_layers=2
        )

        self.fc = nn.Linear(embed_dim,vocab_size)

    def forward(self,src,tgt):

        src = self.embedding(src)
        tgt = self.embedding(tgt)

        src = src.transpose(0,1)
        tgt = tgt.transpose(0,1)

        output = self.transformer(src,tgt)

        output = output.transpose(0,1)

        logits = self.fc(output)

        return logits

In [48]:
vocab_size = len(vocab)
embed_dim = 32

model = SimpleTransformer(vocab_size,embed_dim)

In [49]:
criterion = nn.CrossEntropyLoss()

In [50]:
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [51]:
epochs = 100

for epoch in range(epochs):

    optimizer.zero_grad()

    output = model(inputs,inputs)

    loss = criterion(
        output.reshape(-1,vocab_size),
        targets.reshape(-1)
    )

    loss.backward()

    optimizer.step()

    if epoch%10==0:
        print("Epoch:",epoch,"Loss:",loss.item())

Epoch: 0 Loss: 1.4796156883239746
Epoch: 10 Loss: 0.3982314169406891
Epoch: 20 Loss: 0.4320966899394989
Epoch: 30 Loss: 0.4249226152896881
Epoch: 40 Loss: 0.3241429924964905
Epoch: 50 Loss: 0.4160973131656647
Epoch: 60 Loss: 0.32941558957099915
Epoch: 70 Loss: 0.2957950234413147
Epoch: 80 Loss: 0.330543577671051
Epoch: 90 Loss: 0.34979909658432007


In [52]:
test = torch.tensor([[word_to_idx["hello"],
                      word_to_idx["world"],
                      word_to_idx["hello"]]])

prediction = model(test,test)

predicted_word = torch.argmax(prediction[0,-1]).item()

print(idx_to_word[predicted_word])

world
